# Cost of Living Visualiser — Data Exploration

Exploring the price dataset and comparing salaries with an estimated monthly cost of living (rent, bills, transport, groceries and leisure).

- **Section 1** loads and inspects the raw CSV.
- **Section 2** does all the cleaning and builds every table the charts use.
- **Sections 3–5** only draw charts from those tables, each with notes on what works and what doesn't.

run in terminal and restart kernel
```pip install numpy pandas matplotlib plotly nbformat```

In [ ]:
# Imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display
from statistics import NormalDist

%matplotlib inline
pio.renderers.default = 'iframe_connected'  # show Plotly charts in any Jupyter version

colour_blue = '#0072B2'
colour_green = '#009E73'
colour_orange = '#D55E00'

## 1. Load and inspect the raw CSV

In [ ]:
df = pd.read_csv('../data/europe_allyouneed.csv', encoding='utf-8-sig')  # utf-8-sig removes the hidden character before 'item'

print('Shape:', df.shape)
print('Countries:', df['country'].nunique(), '| Items:', df['item'].nunique())
display(df.head())

In [ ]:
df.info()
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))

**Current Dataset State**

- One row = one item price in one country (long format)
- `price` was read as text because values contain "€" and thousands separators (e.g. `€61,600.59`)
- `price` is missing only for **Minimum Wage**, in 13 countries: 8 have no national statutory minimum wage (e.g. Sweden, Italy, Switzerland) and 5 have one but no value in the data (UK, Turkey, Russia, Kosovo, Bosnia and Herzegovina)
- `price_range` is missing for salary and minimum wage, and contains outliers, so it is not used — the analysis uses `price` (the average)

## 2. Cleaning and preparation

Everything the charts need is built here; the charts in sections 3–5 do not change the data.

| Table | One row per | Used for |
|:--|:--|:--|
| `df` | item price in a country (cleaned) | building the tables below, item-level charts |
| `countries` | country — salary, minimum wage, rents, bills, transport, groceries, leisure, monthly cost, money left and percentages | almost every chart |
| `app` | country — `countries` plus region, affordability ratio and rent shares | section 5 |
| `cost_long` | country × cost part (long format) | pie, treemap and sunburst charts |

### 2.1 Clean the prices

Remove "€" and `,`, then convert to numbers.

In [ ]:
df['price'] = pd.to_numeric(df['price'].str.replace('€', '', regex=False)
                                       .str.replace(',', '', regex=False), errors='coerce')

print('Duplicate rows:', df.duplicated().sum())
display(df.loc[df['price'] == 0, ['item', 'country', 'price']])

**Finding:** Malta's monthly public transport pass is €0. This is plausible rather than an error — Malta made public transport free for residents in 2022 — so the value is kept.

### 2.2 Country table — one row per country

Some items are already monthly prices (rent, utilities, internet, mobile, transport pass). Groceries and leisure are **unit prices**, so they are multiplied by an estimated **monthly quantity for one adult** (the baskets below — our estimate, to be agreed by the group).

monthly cost = rent (1-bed, outside centre) + bills (utilities + internet + mobile) + transport pass + groceries + leisure

money left = net salary − monthly cost

In [ ]:
items = {
    'Average Monthly Net Salary (After Tax)': 'net_salary',
    'Minimum Wage': 'minimum_wage',
    '1 Bedroom Apartment Outside of City Centre': 'rent',
    '1 Bedroom Apartment in City Centre': 'rent_centre',
    'Basic Utilities for 85 m2 Apartment (Electricity, Heating, Cooling, Water, Garbage)': 'utilities',
    'Broadband Internet (Unlimited Data, 60 Mbps or Higher)': 'internet',
    'Mobile Phone Plan (Monthly, with Calls and 10GB+ Data)': 'mobile',
    'Monthly Public Transport Pass (Regular Price)': 'transport',
}

countries = (df[df['item'].isin(items)]
             .pivot(index='country', columns='item', values='price')
             .rename(columns=items)
             .rename_axis(columns=None)
             .reset_index())
countries = countries[['country'] + list(items.values())]

# Monthly quantities for one adult: (category, item): quantity
# The category is included because "Imported Beer" appears in both groceries and eat out.
grocery_basket = {
    ('groceries', 'Milk (Regular, 1 Liter)'): 8,
    ('groceries', 'Fresh White Bread (500 g Loaf)'): 6,
    ('groceries', 'White Rice (1 kg)'): 1.5,
    ('groceries', 'Eggs (12, Large Size)'): 3,
    ('groceries', 'Local Cheese (1 kg)'): 0.75,
    ('groceries', 'Chicken Fillets (1 kg)'): 2.5,
    ('groceries', 'Beef Round or Equivalent Back Leg Red Meat (1 kg)'): 1,
    ('groceries', 'Apples (1 kg)'): 2,
    ('groceries', 'Bananas (1 kg)'): 2,
    ('groceries', 'Oranges (1 kg)'): 2,
    ('groceries', 'Tomatoes (1 kg)'): 2,
    ('groceries', 'Potatoes (1 kg)'): 4,
    ('groceries', 'Onions (1 kg)'): 1,
    ('groceries', 'Lettuce (1 Head)'): 4,
    ('groceries', 'Bottled Water (1.5 Liter)'): 10,
}
leisure_basket = {
    ('sport_and_leis', 'Monthly Fitness Club Membership'): 1,
    ('sport_and_leis', 'Cinema Ticket (International Release)'): 2,
    ('eat out', 'Meal at an Inexpensive Restaurant'): 4,
    ('eat out', 'Cappuccino (Regular Size)'): 8,
}

def basket_cost(basket):
    # price of each item x its monthly quantity, added up per country
    parts = [df[(df['category'] == cat) & (df['item'] == item)].set_index('country')['price'] * qty
             for (cat, item), qty in basket.items()]
    return sum(parts)

countries['groceries'] = countries['country'].map(basket_cost(grocery_basket))
countries['leisure'] = countries['country'].map(basket_cost(leisure_basket))
countries['bills'] = countries['utilities'] + countries['internet'] + countries['mobile']

countries['monthly_cost'] = countries[['rent', 'bills', 'transport', 'groceries', 'leisure']].sum(axis=1)
countries['money_left'] = countries['net_salary'] - countries['monthly_cost']
countries['cost_pct_of_salary'] = (countries['monthly_cost'] / countries['net_salary'] * 100).round(1)
countries['rent_pct_of_salary'] = (countries['rent'] / countries['net_salary'] * 100).round(1)
countries['left_on_min_wage'] = countries['minimum_wage'] - countries['monthly_cost']  # missing where there is no minimum wage
countries['cost_pct_of_min_wage'] = (countries['monthly_cost'] / countries['minimum_wage'] * 100).round(1)
countries['status'] = np.where(countries['money_left'] >= 0, 'Money left', 'Short')

display(countries.sort_values('money_left', ascending=False).round(0))

### 2.3 Regions and extra ratios

- **affordability ratio** = net salary ÷ monthly cost (×2 = the salary covers the costs twice)
- **rent % of minimum wage** and **rent share of cost** (rent ÷ monthly cost)

In [ ]:
regions = {
    'Western': ['Austria', 'Belgium', 'France', 'Germany', 'Ireland', 'Netherlands', 'Switzerland', 'United Kingdom'],
    'Northern & Baltic': ['Denmark', 'Finland', 'Iceland', 'Norway', 'Sweden', 'Estonia', 'Latvia', 'Lithuania'],
    'Southern': ['Croatia', 'Greece', 'Italy', 'Malta', 'Portugal', 'Slovenia', 'Spain', 'Turkey'],
    'Central & Eastern': ['Bulgaria', 'Czech Republic', 'Hungary', 'Poland', 'Romania', 'Russia', 'Slovakia'],
    'Western Balkans': ['Albania', 'Bosnia and Herzegovina', 'Kosovo', 'Montenegro', 'North Macedonia', 'Serbia'],
}
region_of = {c: r for r, cs in regions.items() for c in cs}

app = countries.set_index('country').copy()
app['region'] = app.index.map(region_of)
app['affordability_ratio'] = app['net_salary'] / app['monthly_cost']        # x2 = salary covers costs twice
app['rent_pct_of_min_wage'] = app['rent'] / app['minimum_wage'] * 100
app['rent_share_of_cost'] = app['rent'] / app['monthly_cost'] * 100

part_cols = ['rent', 'bills', 'transport', 'groceries', 'leisure']
part_names = {'rent': 'Rent', 'bills': 'Bills', 'transport': 'Transport', 'groceries': 'Groceries', 'leisure': 'Leisure'}
part_palette = {'rent': colour_blue, 'bills': '#E69F00', 'transport': colour_green, 'groceries': '#CC79A7', 'leisure': '#56B4E9'}

print('Countries without a region:', app.index[app['region'].isna()].tolist())
display(app[['region', 'monthly_cost', 'affordability_ratio', 'rent_pct_of_min_wage', 'rent_share_of_cost']].round(2).head())

### 2.4 Cost parts in long format, labels and colours

`cost_long` is the `countries` table in **long format**: one row per country × cost part instead of one column per part. Pie, treemap and sunburst charts need this shape (one column naming the part, one holding the euros).

The labels and colours here are shared by the charts so each cost part always has the same colour.

In [ ]:
cost_parts = {'rent': 'Rent', 'bills': 'Bills', 'transport': 'Transport',
              'groceries': 'Groceries', 'leisure': 'Leisure & eating out'}
part_colours = {'Rent': colour_blue, 'Bills': '#E69F00', 'Transport': colour_green,
                'Groceries': '#CC79A7', 'Leisure & eating out': '#56B4E9'}

cost_long = (countries.melt(id_vars='country', value_vars=list(cost_parts), var_name='part', value_name='euros')
                      .assign(part=lambda t: t['part'].map(cost_parts)))
print(len(cost_long), 'rows =', countries.shape[0], 'countries x', len(cost_parts), 'parts')
display(cost_long.head(10).round(2))

## 3. Salary, rent and money left

Charts built from the `countries` table.

### Static chart — how much of a salary goes on rent?

Notes:
- This chart is very good for showing rent as a % of salary. It highlights how much of your income rent takes in different countries, since rent is usually the biggest expense.
- A good visualisation option. Ireland is in orange because Irish users would want to compare other countries with their own.

In [ ]:
by_rent = countries.sort_values('rent_pct_of_salary')
colours = [colour_orange if c == 'Ireland' else colour_blue for c in by_rent['country']]

fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(by_rent['country'], by_rent['rent_pct_of_salary'], color=colours)
ax.set_xlim(left=0)
ax.set_xlabel('Rent as % of net monthly salary')
ax.set_title('Rent for a 1-bed outside the city centre as a share of salary')
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

### Interactive chart 1 — money left after monthly costs

Hover over a bar to see the salary and cost behind it.

Notes:
- The ideal place to live for most users would be one that leaves more money over after expenses are deducted.
- Definitely a great visualisation: this is one of the main comparisons a user would want to see, and a bar chart makes it very easy to compare.
- The interaction lets a user see the exact figures, which is exactly what we want.
- About the average net salary: this is a crowd-sourced average of what Numbeo users report earning after tax, mixed across all jobs and sectors. Because it is an average, a few high earners can pull it up, so it is not representative of a typical person, and it is especially unrepresentative for students and recent graduates, who often earn the minimum wage or work part-time. It is useful for comparing countries with each other, but the money left for an individual depends on their own income. This is why we also show the minimum-wage view and plan a "my own salary" option.

In [ ]:
fig_left = px.bar(
    countries.sort_values('money_left'), x='money_left', y='country', orientation='h',
    color='status', color_discrete_map={'Money left': colour_green, 'Short': colour_orange},
    hover_data={'net_salary': ':,.0f', 'monthly_cost': ':,.0f', 'money_left': ':,.0f', 'status': False},
    labels={'money_left': 'Money left (€)', 'country': 'Country',
            'net_salary': 'Net salary (€)', 'monthly_cost': 'Monthly cost (€)'},
    title='Net salary minus rent, bills, transport, groceries and leisure', height=850,
)
fig_left.update_layout(template='plotly_white', showlegend=False)
fig_left.show()

### Interactive chart 2 — do higher salaries come with higher rent?

Each point is a country. Hover for details; drag to zoom, double-click to reset.

Notes:
- Richer economies pay more and charge more. Salaries are high in Switzerland, Norway, Denmark and Ireland because workers there produce more value per hour. Rent can't be traded across borders (you can't import a cheaper flat from Bulgaria), so when local incomes rise, local prices for housing and services rise with them.
- The two groups reflect Europe's East–West income gap. The cluster below €1,800 is mostly Central, Eastern and South-Eastern Europe (the former communist economies, plus Greece, Portugal and Turkey). The cluster above €2,400 is Western and Northern Europe. The empty gap between them shows that wages in the East are rising but haven't caught up with the West yet.
- From this chart a user can easily spot the richer economies. Its weakness is that country names only appear on hover. It is more interesting for a user who is researching the economic picture.
- These explanations are likely reasons, not something the data proves — the chart only shows that salary and rent rise together.

In [ ]:
fig_scatter = px.scatter(
    countries, x='net_salary', y='rent', hover_name='country',
    hover_data={'net_salary': ':,.0f', 'rent': ':,.0f', 'rent_pct_of_salary': ':.1f'},
    labels={'net_salary': 'Net monthly salary (€)', 'rent': 'Rent, 1-bed outside centre (€)',
            'rent_pct_of_salary': 'Rent as % of salary'},
    title='Net salary vs rent by country',
)
fig_scatter.update_traces(marker={'size': 11, 'color': colour_blue})
fig_scatter.update_layout(template='plotly_white')
fig_scatter.show()

print('Correlation:', round(countries['net_salary'].corr(countries['rent']), 2))

### Static chart — can the average salary cover the cost of living?

Grouped bars: net salary next to the estimated monthly cost for each country, both from a zero baseline.

Notes:
- A great visualisation for showing the cost of living, which is the main thing we wanted to show in this project.
- The two colours (blue and amber) make it easy to tell the grouped bars apart.
- There are a lot of countries, so it is slightly hard to read because the bars are compressed.
- Showing the amounts in euro on the axis is useful because you can see the economic differences.
- It would be nice as an interactive chart, and it would be good to also show the cost of living on its own next to the net salary to highlight the countries with the highest costs.

In [ ]:
ordered = countries.sort_values('net_salary', ascending=False).reset_index(drop=True)
x = np.arange(len(ordered))
w = 0.4

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w / 2, ordered['net_salary'], width=w, label='Net monthly salary', color=colour_blue)
ax.bar(x + w / 2, ordered['monthly_cost'], width=w, label='Estimated monthly cost', color='#E69F00')
ax.set_xticks(x)
ax.set_xticklabels(ordered['country'], rotation=60, ha='right')
ax.set_ylabel('€ per month')
ax.set_title('Net salary vs estimated monthly cost of living (single person)')
ax.legend()
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

### Static chart — what makes up the cost of living?

A stacked bar shows the total and its parts: rent, bills (utilities + internet + mobile), transport pass, groceries and leisure. Sorted by total cost; rent at the bottom so the biggest part shares a common baseline.

Notes:
- A good chart, but it can be misleading: at first Iceland looks very expensive, but compared with its salary it isn't as bad. It mainly shows how much things cost in euro, so it is useful for a user saving up to move to one of these countries before finding a job there.

In [ ]:
stack_labels = {'rent': 'Rent (1-bed, outside centre)', 'bills': 'Utilities, internet & phone',
              'transport': 'Transport pass', 'groceries': 'Groceries', 'leisure': 'Leisure & eating out'}
stack_colours = {'rent': colour_blue, 'bills': '#E69F00', 'transport': colour_green,
                'groceries': '#CC79A7', 'leisure': '#56B4E9'}

by_cost = countries.sort_values('monthly_cost')
fig, ax = plt.subplots(figsize=(9, 9))
left = np.zeros(len(by_cost))
for part, label in stack_labels.items():
    ax.barh(by_cost['country'], by_cost[part], left=left, label=label,
            color=stack_colours[part], edgecolor='white', linewidth=0.5)
    left = left + by_cost[part].to_numpy()

ax.set_xlim(left=0)
ax.set_xlabel('Estimated monthly cost (€)')
ax.set_title('Breakdown of estimated monthly living costs')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

### Static chart — how much of the salary does the cost of living take?

Monthly cost as a percentage of net salary, with Ireland highlighted. Above 100% the average salary would not cover the estimated costs.

Notes:
- A great visualisation of the main message of this project.
- It could also be shown as a bar chart.

In [ ]:
by_share = countries.sort_values('cost_pct_of_salary').reset_index(drop=True)
target = by_share.loc[by_share['country'] == 'Ireland'].iloc[0]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(by_share['country'], by_share['cost_pct_of_salary'], marker='o', linestyle='none', color=colour_blue)
ax.scatter([target['country']], [target['cost_pct_of_salary']], color=colour_orange, s=80, zorder=5)
ax.annotate(f"Ireland: {target['cost_pct_of_salary']:.0f}% of salary",
            xy=(target['country'], target['cost_pct_of_salary']), xytext=(25, -45),
            textcoords='offset points', arrowprops={'arrowstyle': '->', 'color': colour_orange})
ax.axhline(100, linestyle='--', color='grey', linewidth=1)
ax.set_ylim(0, max(110, by_share['cost_pct_of_salary'].max() + 10))
plt.setp(ax.get_xticklabels(), rotation=70, ha='right')
ax.set_ylabel('Monthly cost as % of net salary')
ax.set_title('Share of net salary needed for the estimated monthly cost')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.show()

### Static chart — rent in the city centre vs outside

Histograms of the 37 country values for a 1-bed apartment, with the same bins so the two panels can be compared. The dashed line is the median.

Notes:
- This answers one of our group questions: how to represent rent fairly, since it varies between the city centre and outside (and with the size of the home).
- It gives the user a situation to relate to.
- It is more useful for the early moving stage, since it doesn't depend on salary.
- It also shows the gaps in rent between countries.
- Its weakness is that country names aren't shown; an interactive version could add them.

In [ ]:
bins = np.histogram_bin_edges(pd.concat([countries['rent_centre'], countries['rent']]), bins=12)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
axes[0].hist(countries['rent_centre'], bins=bins, color=colour_blue, edgecolor='white')
axes[0].set_title('1-bed apartment, city centre')
axes[1].hist(countries['rent'], bins=bins, color=colour_green, edgecolor='white')
axes[1].set_title('1-bed apartment, outside centre')
for ax, column in zip(axes, ['rent_centre', 'rent']):
    median = countries[column].median()
    ax.axvline(median, linestyle='--', color='grey')
    ax.text(median, ax.get_ylim()[1] * 0.92, f'  median €{median:,.0f}', color='grey')
    ax.set_xlabel('Monthly rent (€)')
    ax.grid(axis='y', alpha=0.2)
axes[0].set_ylabel('Number of countries')
fig.suptitle('Distribution of monthly rent across 37 countries', y=1.02)
fig.tight_layout()
plt.show()

### Static chart — money left on the minimum wage

Minimum wage is missing for 13 countries, so they are **left out rather than set to zero**. The values are most likely gross (before tax) while costs are paid from net income, so this view overstates what a minimum-wage worker has left.

Notes:
- More useful for students, as it uses the minimum wage. It is interesting that in some countries the minimum wage does not cover general expenses.

In [ ]:
min_wage = countries.dropna(subset=['minimum_wage'])
print(f'{len(min_wage)} of {len(countries)} countries have a minimum wage value.')

by_min = min_wage.sort_values('left_on_min_wage')
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(by_min['country'], by_min['left_on_min_wage'],
        color=np.where(by_min['left_on_min_wage'] >= 0, colour_green, colour_orange))
ax.axvline(0, color='grey', linewidth=1)
ax.set_xlabel('Minimum wage minus estimated monthly cost (€)')
ax.set_title('Money left on the minimum wage (minimum wage likely gross)')
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 4. Other chart types

Different chart types for the same prepared data. Every chart below uses the tables from section 2 — no new cleaning.

### Pie charts — how is one person's monthly cost split?

A pie works here because the five parts **add up to the whole monthly cost**. Three countries with very different salaries are compared side by side. Pies are good for "what share is rent?", but poor for comparing exact sizes — the stacked bar above is better for that.

In [ ]:
pie_countries = ['Switzerland', 'Ireland', 'Albania']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, name in zip(axes, pie_countries):
    part = cost_long[cost_long['country'] == name]
    ax.pie(part['euros'], labels=part['part'], colors=[part_colours[p] for p in part['part']],
           autopct='%1.0f%%', startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1})
    total = part['euros'].sum()
    ax.set_title(f'{name}\n€{total:,.0f} per month')
fig.suptitle('Share of the estimated monthly cost by part', y=1.02)
fig.tight_layout()
plt.show()

### Interactive pie — pick any country

Choose a country from the dropdown; the pie and title update together.

In [ ]:
names = sorted(countries['country'])
start = 'Ireland'
labels = list(cost_parts.values())

def pie_values(name):
    row = countries.set_index('country').loc[name]
    return [row[p] for p in cost_parts]

def pie_title(name):
    row = countries.set_index('country').loc[name]
    return f"{name}: €{row['monthly_cost']:,.0f} a month ({row['cost_pct_of_salary']:.0f}% of net salary)"

fig_pie = go.Figure(go.Pie(labels=labels, values=pie_values(start), sort=False,
                           marker={'colors': [part_colours[l] for l in labels]},
                           hovertemplate='%{label}: €%{value:,.0f} (%{percent})<extra></extra>'))
buttons = [dict(label=n, method='update', args=[{'values': [pie_values(n)]}, {'title': {'text': pie_title(n)}}])
           for n in names]
fig_pie.update_layout(template='plotly_white', title={'text': pie_title(start)}, height=500,
                      updatemenus=[dict(buttons=buttons, active=names.index(start), x=0, xanchor='left',
                                        y=1.15, yanchor='top')],
                      margin={'t': 110})
fig_pie.show()

### Dumbbell chart — how much more does the city centre cost?

Each line joins a country's 1-bed rent **outside** the centre to the rent **in** the centre, so the length of the line is the extra cost of living centrally. Sorted by city-centre rent.

Notes:
- A great visualisation for showing how rent changes with location.

In [ ]:
dumb = countries.sort_values('rent_centre')[['country', 'rent', 'rent_centre']].copy()
dumb['extra'] = dumb['rent_centre'] - dumb['rent']

line_x, line_y = [], []
for _, row in dumb.iterrows():
    line_x += [row['rent'], row['rent_centre'], None]   # None breaks the line between countries
    line_y += [row['country'], row['country'], None]

fig_dumb = go.Figure()
fig_dumb.add_trace(go.Scatter(x=line_x, y=line_y, mode='lines', line={'color': '#B8C4CE', 'width': 3},
                              hoverinfo='skip', showlegend=False))
fig_dumb.add_trace(go.Scatter(x=dumb['rent'], y=dumb['country'], mode='markers', name='Outside centre',
                              marker={'color': colour_green, 'size': 10},
                              customdata=dumb[['extra']],
                              hovertemplate='%{y}<br>Outside: €%{x:,.0f}<extra></extra>'))
fig_dumb.add_trace(go.Scatter(x=dumb['rent_centre'], y=dumb['country'], mode='markers', name='City centre',
                              marker={'color': colour_blue, 'size': 10},
                              customdata=dumb[['extra']],
                              hovertemplate='%{y}<br>Centre: €%{x:,.0f}<br>Extra: €%{customdata[0]:,.0f}<extra></extra>'))
fig_dumb.update_layout(template='plotly_white', height=850, title='1-bed rent: outside vs city centre',
                       xaxis={'title': 'Monthly rent (€)', 'rangemode': 'tozero'}, yaxis={'title': ''})
fig_dumb.show()

print('Biggest city-centre premium:')
display(dumb.nlargest(5, 'extra').round(0))

### Treemap vs sunburst — the same data, two hierarchy views

Both use the path **Europe → country → cost part**, sized by euros. Click a country to zoom in (click the centre / path bar to go back).

- **Treemap:** areas are easy to compare; good for spotting the biggest countries and parts.
- **Sunburst:** shows the hierarchy as rings; good for following country → part, harder to compare sizes.

In [ ]:
treemap = px.treemap(cost_long, path=[px.Constant('Europe'), 'country', 'part'], values='euros',
                     color='part', color_discrete_map={**part_colours, '(?)': '#E5E7EB'})
sunburst = px.sunburst(cost_long, path=[px.Constant('Europe'), 'country', 'part'], values='euros',
                       color='part', color_discrete_map={**part_colours, '(?)': '#E5E7EB'})

hierarchy = make_subplots(rows=1, cols=2, specs=[[{'type': 'domain'}, {'type': 'domain'}]],
                          subplot_titles=['Treemap', 'Sunburst'])
hierarchy.add_trace(treemap.data[0], row=1, col=1)
hierarchy.add_trace(sunburst.data[0], row=1, col=2)
hierarchy.update_traces(hovertemplate='%{label}<br>€%{value:,.0f}<extra></extra>')
hierarchy.update_layout(height=650, title='Estimated monthly cost: Europe → country → cost part')
hierarchy.show()

### Parallel coordinates — which countries meet several conditions at once?

Each line is a country, each axis a variable. **Drag along an axis to brush** (filter) — e.g. salary above €2,000 and money left above €1,000. Line colour = money left.

Brushing in the browser does not change the Python data, so the same filter is reproduced with Pandas below.

Notes:
- A very advanced chart; at first glance it wouldn't appeal to the average user.
- It is good for comparing each expense against the range of values (e.g. one country might have a high net salary but a low rent).
- It is limited by the economic gap between countries: the main thing you see is the richest countries, isolated at the top.

In [ ]:
dims = [('net_salary', 'Net salary (€)'), ('rent', 'Rent (€)'), ('groceries', 'Groceries (€)'),
        ('leisure', 'Leisure (€)'), ('monthly_cost', 'Monthly cost (€)'), ('money_left', 'Money left (€)')]

fig_par = go.Figure(go.Parcoords(
    line={'color': countries['money_left'], 'colorscale': 'Cividis', 'showscale': True,
          'colorbar': {'title': 'Money left (€)'}},
    dimensions=[{'label': label, 'values': countries[col]} for col, label in dims],
))
fig_par.update_layout(template='plotly_white', height=500, title='Country profiles across cost variables')
fig_par.show()

matches = countries[(countries['net_salary'] > 2000) & (countries['money_left'] > 1000)]
display(matches[['country', 'net_salary', 'monthly_cost', 'money_left']].sort_values('money_left', ascending=False).round(0))

### Choropleth maps — where does the cost of living take the biggest share of income?

Sequential colour (`Cividis`) because the value only goes one way: higher % = harder to afford. Two versions:
1. **Minimum wage** (gross) — the 13 countries without a minimum wage are grey. Above 100% the minimum wage does not cover the costs.
2. **Average net salary** — all 37 countries.

Notes:
- A geographic chart suits this project well, and the colour gradient makes it easy to understand. The best measure to show on it is the cost of living as a % of income, as that is our main message.

**Limitation:** large countries dominate a map visually, so use the cost-share chart above for exact ranking.

In [ ]:
# Version 1: minimum wage (13 countries have no value)
fig_share_map = px.choropleth(
    countries.dropna(subset=['minimum_wage']), locations='country', locationmode='country names',
    color='cost_pct_of_min_wage',
    color_continuous_scale='Cividis_r', range_color=[50, 150], scope='europe', hover_name='country',
    hover_data={'country': False, 'minimum_wage': ':,.0f', 'monthly_cost': ':,.0f', 'cost_pct_of_min_wage': ':.0f'},
    labels={'cost_pct_of_min_wage': 'Cost % of minimum wage', 'minimum_wage': 'Minimum wage, gross (€)',
            'monthly_cost': 'Monthly cost (€)'},
    title='Estimated monthly cost as % of the minimum wage (gross)',
)
fig_share_map.update_geos(showcountries=True, countrycolor='white', landcolor='#E5E7EB')
fig_share_map.update_layout(template='plotly_white', height=600)
fig_share_map.show()

In [ ]:
# Version 2: average net salary
fig_share_map = px.choropleth(
    countries, locations='country', locationmode='country names', color='cost_pct_of_salary',
    color_continuous_scale='Cividis_r', range_color=[30, 100], scope='europe', hover_name='country',
    hover_data={'country': False, 'net_salary': ':,.0f', 'monthly_cost': ':,.0f', 'cost_pct_of_salary': ':.0f'},
    labels={'cost_pct_of_salary': 'Cost % of salary', 'net_salary': 'Net salary (€)', 'monthly_cost': 'Monthly cost (€)'},
    title='Estimated monthly cost as % of net salary',
)
fig_share_map.update_geos(showcountries=True, countrycolor='white', landcolor='#E5E7EB')
fig_share_map.update_layout(template='plotly_white', height=600)
fig_share_map.show()

### Heatmap — each cost part as a share of salary

Rows are countries (highest total share at the top), columns are cost parts. Darker = a bigger slice of the salary. This shows *which* part makes a country hard to afford — usually rent.

Notes:
- A heavy chart to look at; pie charts would be easier for a user to read, but the heatmap allows an immediate comparison between countries.

In [ ]:
share_table = countries.set_index('country')[list(cost_parts)].div(countries.set_index('country')['net_salary'], axis=0) * 100
share_table = share_table.rename(columns=cost_parts)
share_table = share_table.loc[countries.sort_values('cost_pct_of_salary', ascending=False)['country']].round(1)

fig_share_heat = px.imshow(share_table, aspect='auto', color_continuous_scale='Cividis_r',
                           labels={'x': 'Cost part', 'y': 'Country', 'color': '% of net salary'},
                           title='Cost parts as % of net salary', height=900)
fig_share_heat.update_traces(hovertemplate='%{y}<br>%{x}: %{z:.1f}% of salary<extra></extra>')
fig_share_heat.update_layout(template='plotly_white')
fig_share_heat.show()

### Normal Q-Q plot — is net salary normally distributed across countries?

Do the 37 countries' salaries follow a bell-curve shape (a "normal distribution"), where most values sit in the middle and fewer sit at each end?

Each point is a country. If the points follow the dashed line, the values look roughly normal. Comparing **raw** and **log** salary shows whether a log scale makes the shape more symmetric.

Each country counts once — this is not the distribution of individual people's salaries.

**Finding:** salary across countries is right-skewed (Switzerland is far above the rest); on a log scale it is close to normal. So we use the median for "typical" values. We keep linear axes in charts for users, because a log axis makes large differences look smaller and is hard for a general audience to read. This is an analysis check, not a chart for the final dashboard.

In [ ]:
ordered = countries.sort_values('net_salary').reset_index(drop=True)
n = len(ordered)
probs = [(i - 0.5) / n for i in range(1, n + 1)]
theoretical = [NormalDist().inv_cdf(p) for p in probs]

def standardise(values):
    return (values - values.mean()) / values.std()

raw = standardise(ordered['net_salary'])
logged = standardise(np.log(ordered['net_salary']))

fig_qq = make_subplots(rows=1, cols=2, subplot_titles=['Raw net salary', 'Log net salary'])
for col, values in [(1, raw), (2, logged)]:
    fig_qq.add_trace(go.Scatter(x=theoretical, y=values, mode='markers', text=ordered['country'],
                                marker={'color': colour_blue, 'size': 8}, showlegend=False,
                                hovertemplate='%{text}<br>theoretical %{x:.2f}<br>observed %{y:.2f}<extra></extra>'),
                     row=1, col=col)
    fig_qq.add_trace(go.Scatter(x=[-2.5, 2.5], y=[-2.5, 2.5], mode='lines', showlegend=False,
                                line={'dash': 'dash', 'color': 'grey'}), row=1, col=col)
    fig_qq.update_xaxes(title_text='Theoretical normal quantile', row=1, col=col)
    fig_qq.update_yaxes(title_text='Standardised observed value', row=1, col=col)
fig_qq.update_layout(template='plotly_white', height=450, title='Normal Q-Q plot of net salary')
fig_qq.show()

## 5. Further chart options

More chart ideas proposed by the team, recreated here with the same tables so every option can be compared before choosing the final visualisations. They use the `app` table from section 2.3.

### Can the minimum wage cover the cost of living? (grouped bars)

Like the salary version above, but with the **minimum wage** as income. Labels mark the countries where cost is higher than the wage.

In [ ]:
d = app.dropna(subset=['minimum_wage']).sort_values('minimum_wage')
y = np.arange(len(d))
h = 0.38

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(y + h / 2, d['minimum_wage'], height=h, color=colour_blue, label='Minimum wage (gross)')
cost_bars = ax.barh(y - h / 2, d['monthly_cost'], height=h, color='#E69F00', label='Monthly cost')
ax.bar_label(cost_bars, labels=['cost > wage' if c > w else '' for c, w in zip(d['monthly_cost'], d['minimum_wage'])],
             padding=4, fontsize=8)
ax.set_yticks(y)
ax.set_yticklabels(d.index)
ax.set_xlim(left=0)
ax.set_xlabel('€ per month')
ax.set_title('Can the minimum wage cover the cost of living?')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

### Are countries with higher minimum wages also more expensive? (scatter)

One point per country, colour = rent as % of the monthly cost. Points **below** the dashed line: the wage does not cover the costs. Correlation is an association, not a cause.

In [ ]:
d = app.dropna(subset=['minimum_wage'])
fig, ax = plt.subplots(figsize=(10, 7))
points = ax.scatter(d['monthly_cost'], d['minimum_wage'], c=d['rent_share_of_cost'], cmap='cividis',
                    s=70, edgecolors='white', zorder=3)
fig.colorbar(points, ax=ax, label='Rent as % of monthly cost')
top = max(d['monthly_cost'].max(), d['minimum_wage'].max()) * 1.05
ax.plot([0, top], [0, top], linestyle='--', color='grey', linewidth=1)
for name in ['Ireland', 'Germany', 'Malta', 'Portugal', 'Albania', 'Czech Republic']:
    ax.annotate(name, (d.at[name, 'monthly_cost'], d.at[name, 'minimum_wage']),
                xytext=(6, 4), textcoords='offset points', fontsize=9)
r = d['monthly_cost'].corr(d['minimum_wage'])
ax.set_xlim(0, d['monthly_cost'].max() * 1.1)
ax.set_ylim(0, d['minimum_wage'].max() * 1.1)
ax.set_xlabel('Monthly cost of living (€)')
ax.set_ylabel('Minimum wage, gross (€)')
ax.set_title(f'Minimum wage vs cost of living (Pearson r = {r:.2f})')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

### How much of the minimum wage goes on rent? (bars + 40% line)

The dashed line is Eurostat's 40% housing-cost overburden threshold. It is defined on *disposable household income*, so here it is only a rough reference.

In [ ]:
d = app.dropna(subset=['minimum_wage']).sort_values('rent_pct_of_min_wage')
fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(d.index, d['rent_pct_of_min_wage'],
               color=[colour_orange if v > 40 else colour_blue for v in d['rent_pct_of_min_wage']])
ax.bar_label(bars, labels=[f'{v:.0f}%' for v in d['rent_pct_of_min_wage']], padding=3, fontsize=8)
ax.axvline(40, linestyle='--', color='grey', linewidth=1)
ax.text(41, 0, '40% threshold', color='grey', fontsize=9, va='center')
ax.set_xlim(0, d['rent_pct_of_min_wage'].max() * 1.12)
ax.set_xlabel('Rent (1-bed outside centre) as % of the gross minimum wage')
ax.set_title('How much of the minimum wage goes on rent?')
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()
print('Countries above 40%:', int((d['rent_pct_of_min_wage'] > 40).sum()), 'of', len(d))

### Where does the monthly budget go? (heatmap, % of total cost)

Same idea as the heatmap above, but each part is shown as a share of the **monthly cost** instead of the salary, so every row adds up to 100%. Sorted by rent share.

In [ ]:
share = app[part_cols].div(app['monthly_cost'], axis=0).mul(100).rename(columns=part_names)
share = share.sort_values('Rent', ascending=False)

fig, ax = plt.subplots(figsize=(8, 11))
im = ax.imshow(share.values, cmap='Blues', aspect='auto', vmin=0)
ax.set_xticks(range(share.shape[1]))
ax.set_xticklabels(share.columns)
ax.set_yticks(range(share.shape[0]))
ax.set_yticklabels(share.index)
ax.xaxis.tick_top()
for i in range(share.shape[0]):
    for j in range(share.shape[1]):
        v = share.iat[i, j]
        ax.text(j, i, f'{v:.0f}%', ha='center', va='center', fontsize=8,
                color='white' if v > share.values.max() * 0.55 else 'black')
fig.colorbar(im, ax=ax, fraction=0.04, label='% of monthly cost')
ax.set_title('Share of each part in the monthly cost', pad=30)
fig.tight_layout()
plt.show()

### Where does an average salary go furthest? (affordability ratio map)

**Affordability ratio** = net salary ÷ monthly cost. ×2.0 means the salary covers the costs twice. This is the same information as "cost % of salary", turned upside down.

In [ ]:
fig_ratio = px.choropleth(
    app.reset_index(), locations='country', locationmode='country names', color='affordability_ratio',
    color_continuous_scale='Blues', scope='europe', hover_name='country',
    hover_data={'country': False, 'net_salary': ':,.0f', 'monthly_cost': ':,.0f', 'affordability_ratio': ':.2f'},
    labels={'affordability_ratio': 'Salary ÷ cost', 'net_salary': 'Net salary (€)', 'monthly_cost': 'Monthly cost (€)'},
    title='Average net salary ÷ monthly cost of living',
)
fig_ratio.update_geos(showcountries=True, countrycolor='white', landcolor='#E5E7EB')
fig_ratio.update_layout(template='plotly_white', height=600)
fig_ratio.show()

### Ireland vs the European median (pie charts)

The left pie is one country; the right pie is the median of each part across all 37 countries.

In [ ]:
profile_country = 'Ireland'
median_parts = app[part_cols].median()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, values, title in [(axes[0], app.loc[profile_country, part_cols], f"{profile_country}: €{app.at[profile_country, 'monthly_cost']:,.0f} a month"),
                          (axes[1], median_parts, f'European median: €{median_parts.sum():,.0f} a month')]:
    ax.pie(values, labels=[part_names[p] for p in part_cols], colors=[part_palette[p] for p in part_cols],
           autopct='%1.0f%%', startangle=90, counterclock=False, wedgeprops={'edgecolor': 'white', 'linewidth': 1})
    ax.set_title(title)
fig.suptitle('Where does the monthly budget go?', y=1.02)
fig.tight_layout()
plt.show()

### How is the monthly cost distributed? (histogram, few vs many bins)

The same 37 values with two bin settings. The bin count changes what shape you see, so it is a design choice.

In [ ]:
median_cost = app['monthly_cost'].median()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, bins, title in [(axes[0], 5, 'Few bins: coarse summary'), (axes[1], 15, 'More bins: more detail')]:
    ax.hist(app['monthly_cost'], bins=bins, color='#56B4E9', edgecolor='white')
    ax.axvline(median_cost, linestyle='--', color=colour_orange, label=f'Median €{median_cost:,.0f}')
    ax.set_title(title)
    ax.set_xlabel('Monthly cost of living (€)')
    ax.set_ylabel('Number of countries')
    ax.legend()
    ax.grid(axis='y', alpha=0.2)
fig.suptitle('Distribution of the monthly cost across 37 countries', y=1.02)
fig.tight_layout()
plt.show()

### Does the cost of living differ between regions? (box plot)

Each box summarises the countries in one region; the dots are the individual countries.

In [ ]:
region_order = list(regions)
data_by_region = [app.loc[app['region'] == r, 'monthly_cost'] for r in region_order]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(data_by_region, patch_artist=True, boxprops={'facecolor': '#cde2fb'})
ax.set_xticklabels(region_order)
rng = np.random.default_rng(0)
for i, values in enumerate(data_by_region, start=1):
    ax.scatter(i + rng.uniform(-0.12, 0.12, len(values)), values, s=18, color='grey', zorder=3)
ax.set_ylabel('Monthly cost of living (€)')
ax.set_title('Monthly cost by region')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.show()

### How much do you need to earn to live in most of Europe? (line)

For each net salary, the share of countries where that salary covers the monthly cost. The marked point is the median of the countries' average salaries.

Notes:
- A very high-level view of Europe, not specific enough to be useful to some users.
- Shows the economic differences between European countries.

In [ ]:
salaries = np.arange(400, 4001, 100)
covered = np.array([(app['monthly_cost'] <= s).mean() * 100 for s in salaries])
ref = app['net_salary'].median()
ref_cov = (app['monthly_cost'] <= ref).mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(salaries, covered, marker='o', markersize=4, color=colour_blue)
ax.scatter([ref], [ref_cov], color=colour_orange, s=65, zorder=5)
ax.annotate(f'Median average salary (€{ref:,.0f})\ncovers costs in {ref_cov:.0f}% of countries',
            xy=(ref, ref_cov), xytext=(40, -70), textcoords='offset points',
            arrowprops={'arrowstyle': '->', 'color': colour_orange})
ax.set_ylim(0, 105)
ax.set_xlabel('Monthly net salary (€)')
ax.set_ylabel('% of countries where it covers the cost')
ax.set_title('How much do you need to earn to live in most of Europe?')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

### Three questions, three encodings (dashboard)

Compare (bars), relate (scatter) and distribute (histogram) side by side.

Notes:
- Country names can't be seen in the middle chart, which is a limitation.

In [ ]:
mw = app.dropna(subset=['minimum_wage'])
mw_left = mw['minimum_wage'] - mw['monthly_cost']
top10 = mw_left.sort_values().tail(10)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
axes[0].barh(top10.index, top10.values, color=colour_blue)
axes[0].set_title('Compare | € left on the minimum wage (top 10)')
axes[1].scatter(mw['monthly_cost'], mw['minimum_wage'], color=colour_blue, s=25)
axes[1].set_title('Relate | minimum wage vs cost')
axes[1].set_xlabel('Monthly cost (€)')
axes[1].set_ylabel('Minimum wage (€)')
axes[2].hist(app['monthly_cost'], bins=10, color='#56B4E9', edgecolor='white')
axes[2].set_title('Distribute | monthly cost')
axes[2].set_xlabel('Monthly cost (€)')
axes[2].set_ylabel('Countries')
for ax in axes:
    ax.grid(alpha=0.2)
fig.suptitle('Three questions, three encodings', y=1.03)
fig.tight_layout()
plt.show()

### Which countries become affordable as the salary rises? (animated bars)

Press play or drag the slider: each frame is one net salary, and the bars show money left in every country. Red = the salary does not cover the costs.

Notes:
- Useful for the early moving stage.
- Not very practical, as it uses the same salary for every country and doesn't account for the economic differences that set what people earn there.

In [ ]:
steps = list(range(600, 4201, 200))
by_cost = app.sort_values('monthly_cost')
frames = pd.DataFrame([{'country': c, 'salary': s, 'left': s - v}
                       for s in steps for c, v in by_cost['monthly_cost'].items()])
frames['status'] = np.where(frames['left'] >= 0, 'Money left', 'Short')
lim = frames['left'].abs().max() * 1.05

fig_anim = px.bar(frames, x='left', y='country', orientation='h', animation_frame='salary',
                  color='status', color_discrete_map={'Money left': colour_green, 'Short': colour_orange},
                  range_x=[-lim, lim], category_orders={'country': list(by_cost.index)},
                  labels={'left': 'Money left (€)', 'country': '', 'salary': 'Net salary (€)'},
                  title='Money left at different net salaries', height=850)
fig_anim.update_layout(template='plotly_white', legend_title_text='')
fig_anim.show()

### Everyday prices vs the European median (one country)

Each bar: how much more (+) or less (−) an item costs in the chosen country than the median country. Change `profile_country` in the "Ireland vs the European median" cell above to look at another country.

In [ ]:
price_items = {
    '1-bed rent (outside)': ('renting', '1 Bedroom Apartment Outside of City Centre'),
    '1-bed rent (centre)': ('renting', '1 Bedroom Apartment in City Centre'),
    'Utilities (85 m²)': ('utilities', 'Basic Utilities for 85 m2 Apartment (Electricity, Heating, Cooling, Water, Garbage)'),
    'Internet': ('utilities', 'Broadband Internet (Unlimited Data, 60 Mbps or Higher)'),
    'Transport pass': ('transport', 'Monthly Public Transport Pass (Regular Price)'),
    'Cheap restaurant meal': ('eat out', 'Meal at an Inexpensive Restaurant'),
    'Cappuccino': ('eat out', 'Cappuccino (Regular Size)'),
    'Milk 1 L': ('groceries', 'Milk (Regular, 1 Liter)'),
    'Bread 500 g': ('groceries', 'Fresh White Bread (500 g Loaf)'),
    'Chicken 1 kg': ('groceries', 'Chicken Fillets (1 kg)'),
    'Gym membership': ('sport_and_leis', 'Monthly Fitness Club Membership'),
    'Cinema ticket': ('sport_and_leis', 'Cinema Ticket (International Release)'),
}
rows = []
for label, (cat, item) in price_items.items():
    prices = df[(df['category'] == cat) & (df['item'] == item)].set_index('country')['price']
    rows.append({'item': label, 'price': prices[profile_country], 'median': prices.median(),
                 'diff_pct': (prices[profile_country] / prices.median() - 1) * 100})
vs_median = pd.DataFrame(rows).sort_values('diff_pct')
vs_median['direction'] = np.where(vs_median['diff_pct'] > 0, 'More expensive', 'Cheaper')

fig_vs = px.bar(vs_median, x='diff_pct', y='item', orientation='h', color='direction',
                color_discrete_map={'More expensive': colour_orange, 'Cheaper': colour_blue},
                hover_data={'price': ':,.2f', 'median': ':,.2f', 'diff_pct': ':+.0f', 'direction': False},
                labels={'diff_pct': 'Difference from European median (%)', 'item': '',
                        'price': f'{profile_country} (€)', 'median': 'Median (€)'},
                title=f'{profile_country}: everyday prices vs the European median', height=450)
fig_vs.add_vline(x=0, line_color='grey')
fig_vs.update_layout(template='plotly_white', legend_title_text='')
fig_vs.show()

### Compare two countries (grouped bars)

Change `country_a` and `country_b` to compare any pair.

Notes:
- Very good visualisation that is easy to understand by any user and serves the original purpose of this project: comparing expenses between countries.

In [ ]:
country_a, country_b = 'Ireland', 'Spain'
compare_cols = part_cols + ['monthly_cost']
compare = pd.DataFrame({
    'part': [part_names.get(c, 'Total') for c in compare_cols] * 2,
    'country': [country_a] * len(compare_cols) + [country_b] * len(compare_cols),
    'euros': list(app.loc[country_a, compare_cols]) + list(app.loc[country_b, compare_cols]),
})
fig_cmp = px.bar(compare, x='part', y='euros', color='country', barmode='group',
                 color_discrete_map={country_a: colour_blue, country_b: '#E69F00'},
                 labels={'part': '', 'euros': '€ per month', 'country': ''},
                 title=f'{country_a} vs {country_b}: monthly cost by part', height=450)
fig_cmp.update_traces(hovertemplate='%{fullData.name} · %{x}: €%{y:,.0f}<extra></extra>')
fig_cmp.update_layout(template='plotly_white')
fig_cmp.show()

a, b = app.loc[country_a], app.loc[country_b]
factor = b['monthly_cost'] / a['monthly_cost']
print(f"Earning €{a['net_salary']:,.0f} in {country_a} is like earning €{a['net_salary'] * factor:,.0f} in {country_b} "
      f"({(factor - 1) * 100:+.0f}% cost of living).")

## Notes

- Groceries and leisure use **estimated monthly quantities** for one adult (the baskets in the country table); changing them changes the results.
- Minimum wage is missing for 13 countries and is probably gross (before tax), so money left on the minimum wage is overstated.
- Utilities are quoted for an 85 m² apartment, which is larger than a 1-bed.
- Average net salary is not everyone's income, and national averages hide differences between cities.
- A link between salary and rent is not proof that one causes the other.